In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pandas numpy scikit-learn matplotlib seaborn optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 21.6 MB/s eta 0:00:00


In [ ]:


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score,
    recall_score, roc_auc_score
)

import optuna
import warnings
warnings.filterwarnings("ignore")



np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


DATA_PATH = "/content/drive/MyDrive/WAF_THESIS/DATASETS/merged_all_attacks.csv"
TARGET    = "class"
DROP_COLS = ["attack_type", "class"]

df = pd.read_csv(DATA_PATH).dropna()

print(f"Dataset shape  : {df.shape}")
print(f"Class balance  :\n{df[TARGET].value_counts()}")
print(f"Attack types   :\n{df['attack_type'].value_counts()}\n")


X = df.drop(columns=DROP_COLS).values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

INPUT_DIM = X.shape[1]
print(f"Input features : {INPUT_DIM}")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train samples  : {len(X_train)}")
print(f"Test  samples  : {len(X_test)}\n")

# DataLoaders
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))

train_loader      = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader       = DataLoader(test_ds,  batch_size=512, shuffle=False)
train_eval_loader = DataLoader(train_ds, batch_size=512, shuffle=False)



class TabularMLP(nn.Module):
    def __init__(self, input_dim, h1, h2, h3, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h3, 32),
            nn.ReLU(),
            nn.Dropout(dropout / 2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


def evaluate(model, loader, y_true):
    model.eval()
    preds_prob = []
    with torch.no_grad():
        for X_batch, _ in loader:
            out = torch.sigmoid(model(X_batch.to(device)))
            preds_prob.extend(out.cpu().numpy())

    preds_prob = np.array(preds_prob).squeeze()
    preds_bin  = (preds_prob >= 0.5).astype(int)

    return {
        "Accuracy"  : accuracy_score (y_true, preds_bin),
        "F1"        : f1_score       (y_true, preds_bin, zero_division=0),
        "Precision" : precision_score(y_true, preds_bin, zero_division=0),
        "Recall"    : recall_score   (y_true, preds_bin, zero_division=0),
        "ROC-AUC"   : roc_auc_score  (y_true, preds_prob),
    }


def objective(trial):
    lr           = trial.suggest_float("lr",           1e-4, 1e-2, log=True)
    dropout      = trial.suggest_float("dropout",      0.1,  0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    h1           = trial.suggest_categorical("h1", [128, 256, 512])
    h2           = trial.suggest_categorical("h2", [64,  128, 256])
    h3           = trial.suggest_categorical("h3", [32,  64,  128])

    model     = TabularMLP(INPUT_DIM, h1, h2, h3, dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_f1 = 0.0
    for epoch in range(30):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        preds = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                preds.extend(torch.sigmoid(model(X_batch.to(device))).cpu().numpy())

        preds_bin = (np.array(preds).squeeze() >= 0.5).astype(int)
        f1        = f1_score(y_test, preds_bin, zero_division=0)

        trial.report(f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        best_f1 = max(best_f1, f1)

    return best_f1


print("=" * 60)
print("[Optuna] Hyperparameter tuning — 30 trials")
print("=" * 60)

study = optuna.create_study(
    direction = "maximize",
    pruner    = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\n  Best Optuna F1 : {study.best_value:.4f}")
print(f"  Best Params    : {study.best_params}")




print("\n" + "=" * 60)
print("[Retrain] Training with best hyperparameters (50 epochs)")
print("=" * 60)

bp = study.best_params
model = TabularMLP(
    INPUT_DIM,
    h1=bp["h1"], h2=bp["h2"], h3=bp["h3"],
    dropout=bp["dropout"]
).to(device)

optimizer = optim.Adam(model.parameters(), lr=bp["lr"], weight_decay=bp["weight_decay"])
criterion = nn.BCEWithLogitsLoss()


scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

EPOCHS = 70
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch).squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if epoch % 10 == 0:
        val_metrics = evaluate(model, test_loader, y_test)
        scheduler.step(val_metrics["F1"])
        print(f"  Epoch {epoch:>2}/{EPOCHS}  "
              f"Loss: {running_loss/len(train_loader):.4f}  "
              f"Val-F1: {val_metrics['F1']:.4f}  "
              f"Val-Acc: {val_metrics['Accuracy']:.4f}")



print("\n── Train Metrics ──")
train_m = evaluate(model, train_eval_loader, y_train)
for k, v in train_m.items():
    print(f"  {k:<12}: {v:.4f}")

print("\n── Test Metrics ───")
test_m = evaluate(model, test_loader, y_test)
for k, v in test_m.items():
    print(f"  {k:<12}: {v:.4f}")


print("\n── Per Attack-Type Breakdown (Test Set) ──")

# Rebuild test indices with attack_type info
_, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET])
df_test = df_test.reset_index(drop=True)

model.eval()
all_probs = []
with torch.no_grad():
    X_t = torch.tensor(X_test).to(device)
    for i in range(0, len(X_t), 512):
        out = torch.sigmoid(model(X_t[i:i+512])).cpu().numpy()
        all_probs.extend(out.squeeze())

df_test["pred"] = (np.array(all_probs) >= 0.5).astype(int)

for atype in df_test["attack_type"].unique():
    sub = df_test[df_test["attack_type"] == atype]
    acc = accuracy_score(sub[TARGET], sub["pred"])
    f1  = f1_score(sub[TARGET], sub["pred"], zero_division=0)
    print(f"  {atype:<6}  Accuracy: {acc:.4f}  F1: {f1:.4f}  (n={len(sub)})")


results = {
    "Dataset"         : "merged_all_attacks_Data",
    "Best Optuna F1"  : round(study.best_value, 4),
    "Train Accuracy"  : round(train_m["Accuracy"],  4),
    "Train F1"        : round(train_m["F1"],        4),
    "Train Precision" : round(train_m["Precision"], 4),
    "Train Recall"    : round(train_m["Recall"],    4),
    "Train ROC-AUC"   : round(train_m["ROC-AUC"],   4),
    "Test Accuracy"   : round(test_m["Accuracy"],   4),
    "Test F1"         : round(test_m["F1"],         4),
    "Test Precision"  : round(test_m["Precision"],  4),
    "Test Recall"     : round(test_m["Recall"],     4),
    "Test ROC-AUC"    : round(test_m["ROC-AUC"],    4),
    "Best Params"     : str(study.best_params),
}

pd.DataFrame([results]).to_csv("optuna_mlp_merged_results.csv", index=False)
print("\nResults saved to optuna_mlp_merged_results.csv")

Device: cpu
Dataset shape  : (94210, 21)
Class balance  :
class
1    56452
0    37758
Name: count, dtype: int64
Attack types   :
attack_type
LFI    31177
XSS    30461
OSC    19095
SQL    13477
Name: count, dtype: int64

Input features : 19


[I 2026-06-05 05:43:46,288] A new study created in memory with name: no-name-77736696-ff88-4f2c-b370-e994948a92ae


Train samples  : 75368
Test  samples  : 18842

[Optuna] Hyperparameter tuning — 30 trials


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-05 05:45:07,484] Trial 0 finished with value: 0.9667974778429383 and parameters: {'lr': 0.007593715533400858, 'dropout': 0.3881523192454346, 'weight_decay': 5.586703270136505e-06, 'h1': 128, 'h2': 64, 'h3': 64}. Best is trial 0 with value: 0.9667974778429383.
[I 2026-06-05 05:46:55,159] Trial 1 finished with value: 0.9668488802680303 and parameters: {'lr': 0.006353577414676133, 'dropout': 0.37414708305816236, 'weight_decay': 2.5294354799621397e-05, 'h1': 256, 'h2': 64, 'h3': 32}. Best is trial 1 with value: 0.9668488802680303.
[I 2026-06-05 05:51:49,590] Trial 2 finished with value: 0.9668459571466361 and parameters: {'lr': 0.0029860067128904258, 'dropout': 0.28651361698456723, 'weight_decay': 0.0002717600677721786, 'h1': 256, 'h2': 128, 'h3': 32}. Best is trial 1 with value: 0.9668488802680303.
[I 2026-06-05 05:54:35,986] Trial 3 finished with value: 0.9666607867348739 and parameters: {'lr': 0.0001012882674257537, 'dropout': 0.47479400065646526, 'weight_decay': 5.3086782169